In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
destination_master = pd.read_csv(
    "../data/processed/destination_master.csv"
)

print("Dataset shape:", destination_master.shape)
print("\nColumns:")
print(destination_master.columns.tolist())

display(destination_master.head())

Dataset shape: (76, 12)

Columns:
['Location_Name', 'Located_City', 'Location_Type', 'Average_Rating', 'Review_Count', 'Most_Common_Travel_Month', 'Destination_District', 'Destination_Province', 'Mapped_District', 'Geography_Notes', 'District_Key', 'Accommodation_Count']


,Location_Name,Located_City,Location_Type,Average_Rating,Review_Count,Most_Common_Travel_Month,Destination_District,Destination_Province,Mapped_District,Geography_Notes,District_Key,Accommodation_Count
0,Abhayagiri Dagaba,Anuradhapura,Religious Sites,4.30,96,2,NaN,North Central,Anuradhapura,NaN,anuradhapura,58
1,Ambewela Farms,Nuwara Eliya,Farms,3.90,136,1,NaN,Central,Nuwara Eliya,NaN,nuwara eliya,93
2,Ariyapala Mask Museum,Ambalangoda,Museums,3.87,77,2,Galle,Southern,Galle,NaN,galle,284
3,Arugam Bay,Arugam Bay,Beaches,4.06,187,7,NaN,Eastern,Ampara,NaN,ampara,62
4,Baker's Falls,Nuwara Eliya,Waterfalls,3.96,74,3,NaN,Uva,Nuwara Eliya,NaN,nuwara eliya,93


In [3]:
print("Data types:")
print(destination_master.dtypes)

print("\nMissing values:")
print(destination_master.isnull().sum())

print("\nUnique location types:")
print(destination_master["Location_Type"].unique())

print("\nRating range:")
print(
    destination_master["Average_Rating"].min(),
    destination_master["Average_Rating"].max()
)

Data types:
Location_Name                object
Located_City                 object
Location_Type                object
Average_Rating              float64
Review_Count                  int64
Most_Common_Travel_Month      int64
Destination_District         object
Destination_Province         object
Mapped_District              object
Geography_Notes              object
District_Key                 object
Accommodation_Count           int64
dtype: object

Missing values:
Location_Name                0
Located_City                 0
Location_Type                0
Average_Rating               0
Review_Count                 0
Most_Common_Travel_Month     0
Destination_District        57
Destination_Province         0
Mapped_District              3
Geography_Notes             73
District_Key                 3
Accommodation_Count          0
dtype: int64

Unique location types:
['Religious Sites' 'Farms' 'Museums' 'Beaches' 'Waterfalls' 'Gardens'
 'National Parks' 'Historic Sites' 'Bodies of 

In [4]:
# Create a separate feature dataset
destination_features = destination_master.copy()

# 1. Normalize destination category
destination_features["Location_Type"] = (
    destination_features["Location_Type"]
    .astype(str)
    .str.strip()
)

# 2. Create normalized rating (0 to 1)
destination_features["Rating_Score"] = (
    destination_features["Average_Rating"] - 1
) / 4

# 3. Create normalized review popularity (0 to 1)
destination_features["Review_Popularity"] = (
    destination_features["Review_Count"]
    / destination_features["Review_Count"].max()
)

# 4. Convert travel month to categorical text
destination_features["Travel_Month_Category"] = (
    destination_features["Most_Common_Travel_Month"]
    .astype("Int64")
    .astype(str)
)

# 5. Check results
display(
    destination_features[
        [
            "Location_Name",
            "Location_Type",
            "Average_Rating",
            "Rating_Score",
            "Review_Count",
            "Review_Popularity",
            "Travel_Month_Category"
        ]
    ].head(10)
)

,Location_Name,Location_Type,Average_Rating,Rating_Score,Review_Count,Review_Popularity,Travel_Month_Category
0,Abhayagiri Dagaba,Religious Sites,4.30,0.8250,96,0.260163,2
1,Ambewela Farms,Farms,3.90,0.7250,136,0.368564,1
2,Ariyapala Mask Museum,Museums,3.87,0.7175,77,0.208672,2
3,Arugam Bay,Beaches,4.06,0.7650,187,0.506775,7
4,Baker's Falls,Waterfalls,3.96,0.7400,74,0.200542,3
5,Bentota Beach,Beaches,4.59,0.8975,233,0.631436,1
6,Bluefield Tea Gardens,Farms,4.22,0.8050,314,0.850949,1
7,Brief Garden - Bevis Bawa,Gardens,4.28,0.8200,292,0.791328,2
8,Bundala National Park,National Parks,4.70,0.9250,189,0.512195,2
9,Ceylon Tea Museum,Museums,4.08,0.7700,305,0.826558,2


In [5]:
print("Feature dataset shape:", destination_features.shape)

print("\nMissing values in new features:")
print(
    destination_features[
        [
            "Rating_Score",
            "Review_Popularity",
            "Travel_Month_Category"
        ]
    ].isnull().sum()
)

print("\nRating score range:")
print(
    destination_features["Rating_Score"].min(),
    destination_features["Rating_Score"].max()
)

print("\nReview popularity range:")
print(
    destination_features["Review_Popularity"].min(),
    destination_features["Review_Popularity"].max()
)

Feature dataset shape: (76, 15)

Missing values in new features:
Rating_Score             0
Review_Popularity        0
Travel_Month_Category    0
dtype: int64

Rating score range:
0.53 0.9975

Review popularity range:
0.0948509485094851 1.0


In [6]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

# Encode destination type
type_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

type_encoded = type_encoder.fit_transform(
    destination_features[["Location_Type"]]
)

type_columns = type_encoder.get_feature_names_out(
    ["Location_Type"]
)

type_encoded_df = pd.DataFrame(
    type_encoded,
    columns=type_columns,
    index=destination_features.index
)

print("Encoded destination types:", len(type_columns))
display(type_encoded_df.head())

Encoded destination types: 11


,Location_Type_Beaches,Location_Type_Bodies of Water,Location_Type_Farms,Location_Type_Gardens,Location_Type_Historic Sites,Location_Type_Museums,Location_Type_National Parks,Location_Type_Nature & Wildlife Areas,Location_Type_Religious Sites,Location_Type_Waterfalls,Location_Type_Zoological Gardens
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [7]:
numeric_features = destination_features[
    [
        "Rating_Score",
        "Review_Popularity"
    ]
].copy()

feature_matrix = pd.concat(
    [
        numeric_features,
        type_encoded_df
    ],
    axis=1
)

print("Feature matrix shape:", feature_matrix.shape)
print("Missing values:", feature_matrix.isnull().sum().sum())

display(feature_matrix.head())

Feature matrix shape: (76, 13)
Missing values: 0


,Rating_Score,Review_Popularity,Location_Type_Beaches,Location_Type_Bodies of Water,Location_Type_Farms,Location_Type_Gardens,Location_Type_Historic Sites,Location_Type_Museums,Location_Type_National Parks,Location_Type_Nature & Wildlife Areas,Location_Type_Religious Sites,Location_Type_Waterfalls,Location_Type_Zoological Gardens
0,0.8250,0.260163,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.7250,0.368564,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.7175,0.208672,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.7650,0.506775,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.7400,0.200542,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between all destinations
similarity_matrix = cosine_similarity(feature_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)
print("Missing values:", np.isnan(similarity_matrix).sum())

# Display similarity scores for first 5 destinations
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=destination_features["Location_Name"],
    columns=destination_features["Location_Name"]
)

display(similarity_df.iloc[:5, :5].round(3))

Similarity matrix shape: (76, 76)
Missing values: 0


Location_Name,Abhayagiri Dagaba,Ambewela Farms,Ariyapala Mask Museum,Arugam Bay,Baker's Falls
Location_Name,,,,,
Abhayagiri Dagaba,1.000,0.407,0.392,0.425,0.398
Ambewela Farms,0.407,1.000,0.371,0.424,0.376
Ariyapala Mask Museum,0.392,0.371,1.000,0.386,0.364
Arugam Bay,0.425,0.424,0.386,1.000,0.390
Baker's Falls,0.398,0.376,0.364,0.390,1.000


In [9]:
print("Diagonal values:")
print(np.diag(similarity_matrix)[:10].round(3))

print("Minimum similarity:", similarity_matrix.min().round(3))
print("Maximum similarity:", similarity_matrix.max().round(3))

Diagonal values:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Minimum similarity: 0.306
Maximum similarity: 1.0


In [10]:
# Save engineered destination features
destination_features.to_csv(
    "../data/processed/destination_features.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save feature matrix
feature_matrix.to_csv(
    "../data/processed/destination_feature_matrix.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save cosine similarity matrix
similarity_df.to_csv(
    "../data/processed/destination_similarity_matrix.csv",
    encoding="utf-8-sig"
)

print("Feature engineering outputs saved successfully!")
print("Destinations:", len(destination_features))
print("Feature matrix shape:", feature_matrix.shape)
print("Similarity matrix shape:", similarity_matrix.shape)

Feature engineering outputs saved successfully!
Destinations: 76
Feature matrix shape: (76, 13)
Similarity matrix shape: (76, 76)
